## PreProcessing

In [99]:
import pandas as pd

df = pd.read_csv("../../data/telegram_messages.csv")

df.head(2)

,group_title,message_id,user_id,channel_id,date,text,reply_to,reply_text
0,group_-1001548974049.jsonl,85533,5.538031e+09,1548974049,2026-09-24 14:01:04+00:00,🥳🤬☹️🥳😖🥳😠😠🥳 🇺🇿🇺🇿 Yo‘nalishlar: ✅Toshkent ➡️ Inc...,NaN,NaN
1,group_-1001548974049.jsonl,85532,6.376728e+09,1548974049,2026-09-24 12:42:24+00:00,Assalomu alaykum yaxshimisizlar iphone 16 pro ...,NaN,NaN


In [101]:
X = df.drop(columns=["group_title", "channel_id", "date", "reply_to", "message_id"])

In [100]:
import re
import numpy as np
import urlextract

from collections import Counter
from nltk.stem import PorterStemmer
from sklearn.base import BaseEstimator, TransformerMixin


url_extractor = urlextract.URLExtract()
stemmer = PorterStemmer()


class EmailToWordCounterTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        lower_case=True,
        remove_punctuation=True,
        replace_urls=True,
        replace_numbers=True,
        replace_phone_numbers=True,
        stemming=True
    ):
        self.lower_case = lower_case
        self.remove_punctuation = remove_punctuation
        self.replace_urls = replace_urls
        self.replace_numbers = replace_numbers
        self.replace_phone_numbers = replace_phone_numbers
        self.stemming = stemming

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):

        X_transformed = []

        for message in X:

            # Convert message to string
            text = str(message)

            if self.lower_case:
                text = text.lower()

            # URLs
            if self.replace_urls:
                urls = list(set(url_extractor.find_urls(text)))
                urls.sort(key=len, reverse=True)

                for url in urls:
                    text = text.replace(url, " URL ")

            # Phone numbers
            if self.replace_phone_numbers:
                text = re.sub(
                    r"\+?\d[\d\s().-]{7,}\d",
                    " PHONE ",
                    text
                )

            # Numbers
            if self.replace_numbers:
                text = re.sub(
                    r"\d+(?:\.\d*)?(?:[eE][+-]?\d+)?",
                    " NUMBER ",
                    text
                )

            # Punctuation
            if self.remove_punctuation:
                text = re.sub(r"\W+", " ", text, flags=re.UNICODE)

            # Word counts
            word_counts = Counter(text.split())

            # Stemming
            if self.stemming:
                stemmed_word_counts = Counter()

                for word, count in word_counts.items():
                    stemmed_word = stemmer.stem(word)
                    stemmed_word_counts[stemmed_word] += count

                word_counts = stemmed_word_counts

            X_transformed.append(word_counts)

        return np.array(X_transformed, dtype=object)